In [1]:
import requests
import json
from pathlib import Path
from datetime import datetime
import time
import pandas as pd

# List of entities to fetch
ENTITIES = [
    "games", "staff", "characters", "monsters", "bosses",
    "dungeons", "places", "items"
]

BASE_URL = "https://zelda.fanapis.com/api"
LANDING_DIR = Path("landing")
LOG_FILE = LANDING_DIR / "ingestion_log.jsonl"  # newline-delimited JSON

# Create landing folder if it doesn't exist
LANDING_DIR.mkdir(parents=True, exist_ok=True)

for entity in ENTITIES:
    all_data = []
    page = 1
    start_time = time.time()
    start_timestamp = datetime.now().isoformat()
    output_file = LANDING_DIR / f"{entity}_raw.json"

    print(f"\n=== Fetching entity: {entity} ===")

    try:
        while True:
            url = f"{BASE_URL}/{entity}?page={page}"
            response = requests.get(url, timeout=10)
            response.raise_for_status()

            data = response.json()
            if not data.get("data"):
                break

            all_data.extend(data["data"])
            print(f"Fetched page {page} ({len(data['data'])} records)")
            page += 1

    except requests.exceptions.RequestException as error:
        print(f"API request failed for {entity}:", error)
        # Log failure
        log = {
            "entity": entity,
            "start_time": start_timestamp,
            "end_time": datetime.now().isoformat(),
            "status": "FAILED",
            "error": str(error),
            "rows_fetched": len(all_data),
            "filename": output_file.name,
            "source": f"{BASE_URL}/{entity}"
        }
        with open(LOG_FILE, "a", encoding="utf-8") as log_f:
            log_f.write(json.dumps(log) + "\n")
        continue  # Skip to next entity

    # Save raw JSON (overwrite if exists)
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(all_data, f, indent=4)

    # Compute latest release date if 'released_date' exists
    latest_release_date_str = None
    if entity == "games" and all_data:
        df = pd.DataFrame(all_data)
        df["released_date"] = pd.to_datetime(df["released_date"], errors="coerce")
        latest_release_date = df["released_date"].max()
        latest_release_date_str = latest_release_date.strftime("%Y-%m-%d") if pd.notna(latest_release_date) else None

    end_time = time.time()
    duration = round(end_time - start_time, 2)

    # Log metadata
    log = {
        "entity": entity,
        "start_time": start_timestamp,
        "end_time": datetime.now().isoformat(),
        "duration_seconds": duration,
        "status": "SUCCESS",
        "rows_fetched": len(all_data),
        "pages_fetched": page - 1,
        "filename": output_file.name,
        "source": f"{BASE_URL}/{entity}",
        "latest_release_date": latest_release_date_str
    }

    with open(LOG_FILE, "a", encoding="utf-8") as log_f:
        log_f.write(json.dumps(log) + "\n")

    print(f"Saved {len(all_data)} records to {output_file}")
    if latest_release_date_str:
        print(f"Latest release date: {latest_release_date_str}")



=== Fetching entity: games ===
Fetched page 1 (12 records)
Saved 12 records to landing\games_raw.json
Latest release date: 2020-11-20

=== Fetching entity: staff ===
Fetched page 1 (20 records)
Fetched page 2 (20 records)
Fetched page 3 (20 records)
Fetched page 4 (20 records)
Fetched page 5 (20 records)
Fetched page 6 (20 records)
Fetched page 7 (20 records)
Fetched page 8 (20 records)
Fetched page 9 (20 records)
Fetched page 10 (20 records)
Fetched page 11 (20 records)
Fetched page 12 (14 records)
Saved 234 records to landing\staff_raw.json

=== Fetching entity: characters ===
Fetched page 1 (20 records)
Fetched page 2 (20 records)
Fetched page 3 (20 records)
Fetched page 4 (20 records)
Fetched page 5 (20 records)
Fetched page 6 (20 records)
Fetched page 7 (20 records)
Fetched page 8 (20 records)
Fetched page 9 (20 records)
Fetched page 10 (20 records)
Fetched page 11 (20 records)
Fetched page 12 (20 records)
Fetched page 13 (20 records)
Fetched page 14 (20 records)
Fetched page 15 